# Scalp Analytics — AI Training Pipeline

**Model:** MobileNetV2 Transfer Learning → Klasifikasi Norwood Stage 0–7

**Sebelum mulai:**
1. Runtime → Change runtime type → **T4 GPU**
2. Jalankan cell dari atas ke bawah secara urut

## Cell 1 — Install Library

In [ ]:
!pip install -q albumentations tf2onnx onnxruntime

## Cell 2 — Import Semua Library

In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, Model
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import albumentations as A

print("TensorFlow version:", tf.__version__)
print("GPU tersedia:", tf.config.list_physical_devices('GPU'))

## Cell 3 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATASET_ROOT = "/content/drive/MyDrive/03 - PROJECT/datasets"
MODEL_OUTPUT  = "/content/drive/MyDrive/03 - PROJECT/models"

os.makedirs(MODEL_OUTPUT, exist_ok=True)

print("Dataset path:", DATASET_ROOT)
print("Folders:", os.listdir(DATASET_ROOT))

## Cell 4 — Load Foto + Label dari metadata.json

In [ ]:
LABELED_SOURCES = [
    {"folder": "hf_norwood"},
    {"folder": "hf_bald_men"},
    {"folder": "hf_bald_women"},
]

all_images = []
all_labels = []

for src in LABELED_SOURCES:
    meta_path = Path(DATASET_ROOT) / src["folder"] / "metadata.json"
    img_dir   = Path(DATASET_ROOT) / src["folder"] / "images"

    if not meta_path.exists():
        print(f"[SKIP] {src['folder']} — metadata.json tidak ditemukan")
        continue

    with open(meta_path) as f:
        metadata = json.load(f)

    loaded = 0
    for item in metadata:
        if "error" in item:
            continue
        img_path = img_dir / item["file"]
        if not img_path.exists():
            continue
        try:
            img = Image.open(img_path).convert("RGB").resize((224, 224))
            all_images.append(np.array(img, dtype=np.float32) / 255.0)
            label = min(int(item["label"]), 7)
            all_labels.append(label)
            loaded += 1
        except Exception as e:
            print(f"[ERROR] {img_path.name}: {e}")

    print(f"[OK] {src['folder']}: {loaded} foto dimuat")

X = np.array(all_images)
y = np.array(all_labels)

print(f"\nTotal foto: {len(X)}")
print("Distribusi label:")
for stage in range(8):
    count = np.sum(y == stage)
    bar = "█" * count
    print(f"  Stage {stage}: {count} foto  {bar}")

## Cell 5 — Augmentasi (perbanyak foto 5x)

In [ ]:
augment = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.4),
    A.GaussNoise(p=0.2),
    A.RandomResizedCrop(height=224, width=224, scale=(0.8, 1.0), p=0.3),
])

AUGMENT_MULTIPLIER = 5

X_aug, y_aug = [], []

for img, label in zip(X, y):
    img_uint8 = (img * 255).astype(np.uint8)
    X_aug.append(img)
    y_aug.append(label)
    for _ in range(AUGMENT_MULTIPLIER - 1):
        result = augment(image=img_uint8)["image"]
        X_aug.append(result.astype(np.float32) / 255.0)
        y_aug.append(label)

X_aug = np.array(X_aug)
y_aug = np.array(y_aug)

idx = np.random.permutation(len(X_aug))
X_aug, y_aug = X_aug[idx], y_aug[idx]

print(f"Total setelah augmentasi: {len(X_aug)} foto")

## Cell 6 — Split Train / Validation / Test

In [ ]:
NUM_CLASSES = 8

y_onehot = tf.keras.utils.to_categorical(y_aug, NUM_CLASSES)

X_train, X_temp, y_train, y_temp = train_test_split(
    X_aug, y_onehot, test_size=0.30, random_state=42, stratify=y_aug
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print(f"Train:      {len(X_train)} foto")
print(f"Validation: {len(X_val)} foto")
print(f"Test:       {len(X_test)} foto")

## Cell 7 — Build Model

In [ ]:
base = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)
base.trainable = False

x = base.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.5)(x)
output = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = Model(base.input, output)
print(f"Total params: {model.count_params():,}")
print(f"Trainable:    {sum(tf.size(w).numpy() for w in model.trainable_weights):,}")

## Cell 8 — Training Fase 1 (top layers only)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
]

history1 = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

print("Fase 1 selesai.")

## Cell 9 — Fine-tuning Fase 2

In [ ]:
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history2 = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=25,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

print("Fine-tuning selesai.")

## Cell 10 — Evaluasi & Confusion Matrix

In [ ]:
loss_val, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {acc*100:.1f}%")
print(f"Test Loss:     {loss_val:.4f}")

y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = np.argmax(y_test, axis=1)

top2 = np.mean([y_true[i] in np.argsort(y_pred_prob[i])[-2:] for i in range(len(y_true))])
print(f"Top-2 Accuracy: {top2*100:.1f}%")

STAGE_NAMES = [f"stage_{i}" for i in range(8)]
print("\n" + classification_report(y_true, y_pred, target_names=STAGE_NAMES, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=STAGE_NAMES, yticklabels=STAGE_NAMES)
plt.title("Confusion Matrix — Norwood Stage")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.tight_layout()
plt.savefig(f"{MODEL_OUTPUT}/confusion_matrix.png", dpi=150)
plt.show()

## Cell 11 — Plot Training History

In [ ]:
acc_all     = history1.history["accuracy"]     + history2.history["accuracy"]
val_acc_all = history1.history["val_accuracy"] + history2.history["val_accuracy"]
loss_all    = history1.history["loss"]         + history2.history["loss"]
val_loss_all= history1.history["val_loss"]     + history2.history["val_loss"]
split_ep    = len(history1.history["accuracy"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(acc_all, label="Train")
ax1.plot(val_acc_all, label="Validation")
ax1.axvline(split_ep, color="red", linestyle="--", alpha=0.5, label="Fine-tune start")
ax1.set_title("Accuracy")
ax1.legend()

ax2.plot(loss_all, label="Train")
ax2.plot(val_loss_all, label="Validation")
ax2.axvline(split_ep, color="red", linestyle="--", alpha=0.5, label="Fine-tune start")
ax2.set_title("Loss")
ax2.legend()

plt.tight_layout()
plt.savefig(f"{MODEL_OUTPUT}/training_history.png", dpi=150)
plt.show()

## Cell 12 — Simpan Model (.h5 dan .onnx)

In [ ]:
import tf2onnx
import onnx

h5_path   = f"{MODEL_OUTPUT}/scalp_model.h5"
onnx_path = f"{MODEL_OUTPUT}/scalp_model.onnx"

model.save(h5_path)
print(f"Saved: {h5_path}")

input_signature = [tf.TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name="input")]
model_proto, _ = tf2onnx.convert.from_keras(model, input_signature=input_signature, opset=13)
onnx.save(model_proto, onnx_path)
print(f"Saved: {onnx_path}")

## Cell 13 — Test Inferensi 1 Foto

In [ ]:
import onnxruntime as ort

session = ort.InferenceSession(onnx_path)
STAGES  = [f"stage_{i}" for i in range(8)]

sample_img  = X_test[0:1]
sample_true = STAGES[int(np.argmax(y_test[0]))]

outputs     = session.run(None, {"input": sample_img})[0][0]
pred_stage  = STAGES[int(np.argmax(outputs))]
confidence  = float(np.max(outputs))

print(f"Actual:     {sample_true}")
print(f"Predicted:  {pred_stage}")
print(f"Confidence: {confidence*100:.1f}%")
print("\nProbabilities:")
for stage, prob in zip(STAGES, outputs):
    bar = "█" * int(prob * 30)
    print(f"  {stage}: {prob:.3f}  {bar}")

plt.imshow(sample_img[0])
plt.title(f"Actual: {sample_true}  |  Predicted: {pred_stage} ({confidence*100:.1f}%)")
plt.axis("off")
plt.show()

## Cell 14 — Download Model ke Lokal

Setelah download, taruh file di:
`Backend/app/infrastructure/ai/models/scalp_model.onnx`

In [ ]:
from google.colab import files
files.download(onnx_path)
print("Download selesai!")